# Akuisisi Data Akomodasi di Kecamatan Kintamani
BIFP-243 Akuisisi dan Manajemen Data

Data © OpenStreetMap contributors (lisensi ODbL).

**Isi dulu:** NAMA dan NIM di sel Setup. Jalankan sel dari atas ke bawah, jangan menjalankan sel query berulang-ulang.

## 0. Setup

In [ ]:
import json, time, hashlib
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import requests
import matplotlib.pyplot as plt

NAMA = "ISI_NAMA"
NIM = "ISI_NIM"
EMAIL = "ISI_EMAIL_KAMPUS"

URL = "https://overpass-api.de/api/interpreter"
HEADERS = {"User-Agent": f"INSTIKI-BIFP243-Praktikum/1.0 ({EMAIL})"}

for d in ["data/raw", "data/interim", "docs", "reports", "notebooks"]:
    Path(d).mkdir(parents=True, exist_ok=True)

## 1. Query final
Query ini harus sudah diuji di overpass-turbo.eu (hari 2): pastikan batas wilayah benar-benar Kecamatan Kintamani (`admin_level`), dan simpan tangkapan layar petanya.
Jika batas administratif tidak ditemukan, ganti `area...` dengan bounding box `(selatan, barat, utara, timur)`.

In [ ]:
QUERY = """[out:json][timeout:90];
area["name"="Kintamani"]["boundary"="administrative"]["admin_level"="6"]->.kec;
nwr["tourism"~"^(hotel|guest_house|hostel|motel|apartment|chalet)$"](area.kec);
out center tags;"""
Path("docs/query_overpass.txt").write_text(QUERY, encoding="utf-8")
print(QUERY)

## 2. Akuisisi dengan retry dan timeout

In [ ]:
def ambil_overpass(query, maks_percobaan=4, jeda=5):
    for percobaan in range(1, maks_percobaan + 1):
        try:
            r = requests.post(URL, data={"data": query}, headers=HEADERS, timeout=120)
            if r.status_code == 200:
                return r.json()
            print(f"[percobaan {percobaan}] status {r.status_code}, tunggu {jeda}s")
        except (requests.Timeout, requests.ConnectionError) as e:
            print(f"[percobaan {percobaan}] {type(e).__name__}, tunggu {jeda}s")
        time.sleep(jeda)
        jeda *= 2
    raise RuntimeError("Gagal mengambil data dari Overpass")

waktu_utc = datetime.now(timezone.utc)
data = ambil_overpass(QUERY)
print("Jumlah elemen:", len(data["elements"]))

## 3. Simpan respons mentah (tidak boleh diubah setelah ini) dan provenance

In [ ]:
stamp = waktu_utc.strftime("%Y%m%d_%H%M%S")
raw_path = Path(f"data/raw/osm_akomodasi_kintamani_{stamp}.json")
raw_path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

md5 = hashlib.md5(raw_path.read_bytes()).hexdigest()
provenance = {
    "url_endpoint": URL,
    "query": QUERY,
    "waktu_akuisisi_utc": waktu_utc.isoformat(),
    "jumlah_elemen": len(data["elements"]),
    "berkas_mentah": raw_path.name,
    "md5_berkas_mentah": md5,
    "lisensi": "ODbL - (c) OpenStreetMap contributors",
    "pengambil": {"nama": NAMA, "nim": NIM},
}
Path("docs/provenance.json").write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(provenance, ensure_ascii=False, indent=2)[:800])

## 4. Meratakan menjadi tabel
Telepon dan email sengaja tidak diambil (minimisasi data).

In [ ]:
def ratakan(elements):
    baris = []
    for e in elements:
        tg = e.get("tags", {})
        pusat = e.get("center", e)
        baris.append({
            "osm_type": e["type"], "osm_id": e["id"],
            "nama": tg.get("name"), "jenis": tg.get("tourism"),
            "lat": pusat.get("lat"), "lon": pusat.get("lon"),
            "desa": tg.get("addr:village"),
            "bintang": tg.get("stars"), "jumlah_kamar": tg.get("rooms"),
            "website": tg.get("website") or tg.get("contact:website"),
            "url_osm": f"https://www.openstreetmap.org/{e['type']}/{e['id']}",
        })
    return pd.DataFrame(baris)

akomodasi = ratakan(data["elements"])
print(len(akomodasi), "baris")
print(akomodasi["jenis"].value_counts())
print("Tanpa nama:", akomodasi["nama"].isna().sum())
akomodasi.head()

## 5. Pemeriksaan dasar
Jangan menghapus data diam-diam. Catat setiap keputusan di laporan.

In [ ]:
# 1) duplikat
dup = akomodasi.duplicated(subset=["osm_type", "osm_id"]).sum()
print("Duplikat (osm_type+osm_id):", dup)

# 2) tanpa nama
print("Tanpa nama:", akomodasi["nama"].isna().sum())

# 3) koordinat kosong / di luar kotak kasar Kintamani
# CATATAN: batas di bawah hanya perkiraan kasar. Sesuaikan dengan peta overpass-turbo Anda.
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = -8.36, -8.10, 115.25, 115.50
print("Koordinat kosong:", akomodasi[["lat", "lon"]].isna().any(axis=1).sum())
luar = akomodasi[(akomodasi.lat < LAT_MIN) | (akomodasi.lat > LAT_MAX) |
                 (akomodasi.lon < LON_MIN) | (akomodasi.lon > LON_MAX)]
print("Di luar kotak:", len(luar))

# 4) kolom hampir kosong
print("\nPersen kosong per kolom:")
print((akomodasi.isna().mean() * 100).round(1))

In [ ]:
akomodasi.to_csv("data/interim/akomodasi_kintamani.csv", index=False, encoding="utf-8")
print("Tersimpan:", len(akomodasi), "baris")

## 6. Ringkasan dan peta sebaran

In [ ]:
print(akomodasi["jenis"].value_counts())
print()
print(akomodasi["desa"].value_counts(dropna=False))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
for jenis, g in akomodasi.groupby("jenis"):
    ax.scatter(g["lon"], g["lat"], s=14, label=jenis, alpha=0.8)
ax.set_xlabel("Bujur"); ax.set_ylabel("Lintang")
ax.set_title("Sebaran akomodasi Kecamatan Kintamani\n(c) OpenStreetMap contributors")
ax.legend()
fig.savefig("reports/peta_sebaran.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Uji kelengkapan terhadap BPS
Isi `JUMLAH_BPS` dan `TAHUN_BPS` dari publikasi resmi (Kecamatan Kintamani Dalam Angka / Bangli Dalam Angka / Satu Data Bali). Jika angka per kecamatan tidak ditemukan, jelaskan di laporan.

In [ ]:
JUMLAH_BPS = None   # contoh: 120
TAHUN_BPS = None    # contoh: 2024
SUMBER_BPS = ""     # judul publikasi + halaman/tabel

jumlah_osm = len(akomodasi)
persen = round(jumlah_osm / JUMLAH_BPS * 100, 1) if JUMLAH_BPS else None
banding = pd.DataFrame([{"jumlah_osm": jumlah_osm, "jumlah_bps": JUMLAH_BPS,
                         "tahun_bps": TAHUN_BPS, "persen_kelengkapan": persen,
                         "sumber_bps": SUMBER_BPS}])
banding.to_csv("docs/perbandingan_bps.csv", index=False)
banding

## 8. Sampel 10 data untuk verifikasi manual
Verifikasi nama, jenis, dan lokasi lewat situs resmi akomodasi atau peta (manual, tidak otomatis).

In [ ]:
sampel = akomodasi.sample(n=min(10, len(akomodasi)), random_state=42)[["osm_type","osm_id","nama","jenis","lat","lon","url_osm"]].copy()
sampel["nama_benar"] = ""
sampel["jenis_benar"] = ""
sampel["lokasi_benar"] = ""
sampel["catatan"] = ""
sampel.to_csv("docs/verifikasi_sampel.csv", index=False)
sampel

## 9. Data dictionary dan Data Ethics Checklist

In [ ]:
dd = pd.DataFrame([
 ["osm_type","teks","Jenis objek OSM: node, way, atau relation","way","Tidak"],
 ["osm_id","bilangan bulat","ID objek di OpenStreetMap","123456789","Tidak"],
 ["nama","teks","Nama akomodasi sesuai tag name","(nama akomodasi)","Ya, wajib dihitung jumlahnya"],
 ["jenis","teks","Nilai tag tourism","guest_house","Tidak"],
 ["lat","desimal","Lintang titik atau titik tengah bangunan (WGS84)","-8.27","Tidak"],
 ["lon","desimal","Bujur titik atau titik tengah bangunan (WGS84)","115.37","Tidak"],
 ["desa","teks","Desa sesuai tag addr:village","(nama desa)","Ya"],
 ["bintang","teks","Klasifikasi bintang (tag stars)","3","Ya"],
 ["jumlah_kamar","teks","Jumlah kamar (tag rooms)","20","Ya"],
 ["website","teks","Situs resmi akomodasi","https://...","Ya"],
 ["url_osm","teks","Tautan ke objek di openstreetmap.org untuk pengecekan","https://www.openstreetmap.org/way/...","Tidak"],
], columns=["kolom","tipe","deskripsi","contoh","boleh_kosong"])
dd.to_csv("docs/data_dictionary.csv", index=False)

etika = pd.DataFrame([
 ["Sumber data diizinkan (OSM Overpass + BPS), tidak memakai OTA/Google Maps","", "Lihat docs/provenance.json"],
 ["Atribusi (c) OpenStreetMap contributors ada di README dan laporan","", "Tautkan bagian README/laporan"],
 ["Lisensi ODbL dipahami dan dicatat","", "docs/provenance.json"],
 ["Tidak mengambil data kontak pribadi (telepon, email, media sosial)","", "Fungsi ratakan() tidak memuat kolom tsb"],
 ["Request dibatasi: query dijalankan 1-2 kali, ada retry dengan jeda","", "Sel akuisisi di notebook"],
 ["User-Agent mengidentifikasi pengambil data","", "Sel setup di notebook"],
 ["Berkas data/raw tidak diubah (hash MD5 dicatat)","", "docs/provenance.json"],
 ["Tidak ada penghapusan data diam-diam; keputusan cleaning dicatat","", "Bagian 4 laporan"],
], columns=["butir","status (Ya/Tidak)","bukti"])
etika.to_csv("docs/data_ethics_checklist.csv", index=False)
etika

## 10. Otomatis: checklist etika, README, dan laporan PDF
Sel-sel di bawah mengisi dokumen memakai **angka asli hasil notebook ini**. Jalankan ulang sel 10-12 setelah Anda mengisi angka BPS (sel 7) dan hasil verifikasi sampel (`docs/verifikasi_sampel.csv`).
Sebelum mengumpulkan, baca isi PDF dan pastikan semua pernyataan benar.

In [ ]:
# Checklist etika: dicentang berdasarkan apa yang dilakukan notebook ini
etika["status (Ya/Tidak)"] = "Ya"
etika.to_csv("docs/data_ethics_checklist.csv", index=False)
print("Periksa kembali setiap butir sebelum mengumpulkan, terutama: query hanya dijalankan 1-2 kali.")
etika

In [ ]:
tgl = waktu_utc.strftime("%d %B %Y")
bps_txt = (f"{SUMBER_BPS} (tahun {TAHUN_BPS})" if JUMLAH_BPS else "angka BPS per kecamatan belum diperoleh (lihat laporan)")
readme = f"""# Akuisisi Data Akomodasi di Kecamatan Kintamani

Tugas Praktik BIFP-243 Akuisisi dan Manajemen Data, INSTIKI
Nama: {NAMA} | NIM: {NIM}

## Tujuan
Mengakuisisi data akomodasi (hotel, guest house, hostel, motel, apartemen, chalet) di Kecamatan Kintamani dari API publik, menyimpan respons mentah, mengubahnya menjadi dataset tabular terdokumentasi, dan menilai kelengkapannya terhadap data resmi BPS.

## Sumber data
- OpenStreetMap melalui Overpass API ({URL}), lisensi ODbL.
- Pembanding: {bps_txt}.

## Tanggal akses
{tgl} (waktu UTC lengkap di docs/provenance.json). Jumlah elemen yang diambil: {len(data["elements"])}.

## Metode akuisisi
1. Query Overpass untuk tag tourism hotel, guest_house, hostel, motel, apartment, chalet di dalam batas administratif Kecamatan Kintamani (docs/query_overpass.txt).
2. Pengambilan dengan Python (requests), memakai timeout dan retry berjeda untuk status 429/504.
3. Respons mentah disimpan apa adanya di data/raw/; hash MD5 dicatat di provenance.
4. Respons diratakan menjadi data/interim/akomodasi_kintamani.csv.
5. Pemeriksaan: duplikat, nama kosong, koordinat di luar wilayah, kolom hampir kosong.

## Batasan etika
- Hanya memakai sumber yang diizinkan; tidak memakai OTA maupun Google Maps.
- Data kontak (telepon, email, media sosial) tidak diambil.
- Query dijalankan seperlunya dengan User-Agent yang mengidentifikasi pengambil data.

## Atribusi
Data (c) OpenStreetMap contributors, tersedia di bawah lisensi ODbL.

## Cara menjalankan ulang
1. Buka notebooks/akuisisi_akomodasi.ipynb di Google Colab atau Jupyter.
2. Isi NAMA, NIM, EMAIL di sel Setup.
3. Jalankan semua sel dari atas ke bawah.
4. Hasil ada di data/, docs/, dan reports/.

## Struktur
data/raw (respons mentah), data/interim (CSV), notebooks, docs (query, provenance, data dictionary, checklist etika, perbandingan BPS), reports (laporan PDF).
"""
Path("README.md").write_text(readme, encoding="utf-8")
print("README.md dibuat")

In [ ]:
!pip -q install fpdf2
from fpdf import FPDF

def aman(t):
    return t.replace("\u2013","-").replace("\u2014","-").encode("latin-1","replace").decode("latin-1")

pdf = FPDF()
pdf.set_auto_page_break(auto=True, margin=15)
pdf.add_page()
def judul(t):
    pdf.set_font("Helvetica","B",15); pdf.multi_cell(0,8,aman(t),new_x="LMARGIN",new_y="NEXT"); pdf.ln(1)
def h(t):
    pdf.ln(2); pdf.set_font("Helvetica","B",12); pdf.cell(0,8,aman(t),new_x="LMARGIN",new_y="NEXT")
def para(t):
    pdf.set_font("Helvetica","",10.5); pdf.multi_cell(0,5.5,aman(t),new_x="LMARGIN",new_y="NEXT"); pdf.ln(1)

total = len(akomodasi)
per_jenis = akomodasi["jenis"].value_counts()
per_desa = akomodasi["desa"].fillna("(desa tidak tercatat)").value_counts()
n_nama_kosong = int(akomodasi["nama"].isna().sum())
kosong = (akomodasi.isna().mean() * 100).round(1)

judul("Laporan Akuisisi Data Akomodasi di Kecamatan Kintamani")
para(f"BIFP-243 Akuisisi dan Manajemen Data | Nama: {NAMA} | NIM: {NIM}\nData (c) OpenStreetMap contributors (lisensi ODbL). Tanggal akses: {tgl}.")

h("1. Ringkasan dataset")
para(f"Query Overpass pada tanggal {tgl} mengembalikan {total} akomodasi di Kecamatan Kintamani. Jenis yang dimasukkan: hotel, guest_house, hostel, motel, apartment, dan chalet. Villa di OpenStreetMap umumnya diberi tag guest_house, chalet, atau apartment, sehingga ketiga jenis tersebut dimasukkan agar villa ikut terambil.")
para("Jumlah per jenis: " + "; ".join(f"{k}: {v}" for k, v in per_jenis.items()) + ".")
para("Jumlah per desa (10 teratas): " + "; ".join(f"{k}: {v}" for k, v in per_desa.head(10).items()) + ".")

h("2. Peta sebaran")
pdf.image("reports/peta_sebaran.png", w=110)
para("Gambar: sebaran titik akomodasi berdasarkan koordinat OSM (c) OpenStreetMap contributors.")

h("3. Uji kelengkapan terhadap BPS dan verifikasi sampel")
if JUMLAH_BPS:
    para(f"Sumber pembanding: {SUMBER_BPS}, tahun {TAHUN_BPS}. Jumlah akomodasi menurut BPS: {JUMLAH_BPS}. Jumlah dari OSM: {total}. Persentase kelengkapan = {total} / {JUMLAH_BPS} x 100% = {persen}%.")
else:
    para("Angka BPS per kecamatan untuk Kecamatan Kintamani belum diperoleh, sehingga persentase kelengkapan belum dapat dihitung.")
try:
    vs = pd.read_csv("docs/verifikasi_sampel.csv")
    terisi = vs["nama_benar"].fillna("").astype(str).str.strip() != ""
    if terisi.sum() > 0:
        ok = ((vs["nama_benar"].astype(str).str.lower()=="ya") & (vs["jenis_benar"].astype(str).str.lower()=="ya") & (vs["lokasi_benar"].astype(str).str.lower()=="ya")).sum()
        para(f"Verifikasi manual {int(terisi.sum())} sampel acak: {int(ok)} sampel benar pada nama, jenis, dan lokasi sekaligus.")
    else:
        para("Verifikasi manual 10 sampel belum dilakukan.")
except Exception:
    para("Verifikasi manual 10 sampel belum dilakukan.")

h("4. Masalah kualitas data dan keputusan")
para(f"- Duplikat berdasarkan osm_type + osm_id: {dup}. Tidak ada data yang dihapus diam-diam.\n- Akomodasi tanpa nama: {n_nama_kosong} dari {total}. Tetap disimpan dan dihitung.\n- Koordinat kosong: {int(akomodasi[['lat','lon']].isna().any(axis=1).sum())}; koordinat di luar kotak perkiraan wilayah: {len(luar)}.\n- Persentase kosong: desa {kosong['desa']}%, bintang {kosong['bintang']}%, jumlah_kamar {kosong['jumlah_kamar']}%, website {kosong['website']}%. Kolom bintang dan jumlah_kamar dibiarkan bertipe teks karena isian relawan tidak selalu angka.\n- Data kontak (telepon, email) sengaja tidak diambil.")

h("5. Refleksi")
para("Data akomodasi dari OpenStreetMap tidak boleh dianggap lengkap karena seluruhnya diisi dan diperbarui oleh relawan. Tidak ada kewajiban bagi pemilik usaha atau pemeta untuk mendaftarkan setiap bangunan, sehingga akomodasi kecil seperti homestay, kamar sewa di rumah warga, dan villa baru sering belum tercatat. Kelengkapan juga bergantung pada aktivitas relawan di suatu wilayah, sehingga kawasan yang ramai wisatawan bisa lebih lengkap dibanding desa yang jauh. Atribut seperti nama, jumlah kamar, dan bintang sering kosong atau tidak seragam. Pembatasan pada nilai tag tourism tertentu juga membuat akomodasi dengan tag lain terlewat. Karena itu jumlah OSM lebih tepat dibaca sebagai batas bawah, bukan angka sebenarnya. Untuk riset lanjutan, perlu ditambahkan data resmi BPS dan dinas pariwisata daerah, data perizinan usaha, data pemerintah desa, serta verifikasi lapangan atau survei sampel untuk menaksir akomodasi yang belum terpeta.")

h("Atribusi")
para("Data (c) OpenStreetMap contributors, tersedia di bawah lisensi ODbL.")
pdf.output("reports/laporan_akomodasi_kintamani.pdf")
print("Laporan PDF dibuat, halaman:", pdf.page_no())

In [ ]:
# Unduh hasil (Colab). Simpan notebook ini sendiri lewat File > Download .ipynb, lalu taruh di folder notebooks/
!zip -rq hasil.zip README.md data docs reports
from google.colab import files
files.download("hasil.zip")